# YOLO11 — Plate Detector (biển số)

Notebook train YOLO11n (Ultralytics ≥ 8.4) cho detector biển số xe. Input là
dataset portable xuất từ R8 với manifest `human_curated_group_disjoint`.

Kaggle path: `/kaggle/input/<dataset-slug>/` chứa ảnh + label + data.yaml +
manifest.json.

Pre-flight sẽ kiểm:
- manifest.json hợp lệ + hash khớp.
- Đủ train/val/test với count > 0.
- holdout_status = human_curated_group_disjoint.

Resume: từ `last.pt` trong save_dir. Không ép upgrade Torch local.


In [ ]:
import os
# Kaggle environment: pin Ultralytics, torch, fast_plate_ocr theo kaggle image.
# KHÔNG ép nâng cấp Torch local — pin đúng version đang dùng (YOLO11 cần ≥8.4).
os.environ.setdefault("YOLO_CONFIG_DIR", "/kaggle/working/yolo_cfg")
os.environ.setdefault("ULTRALYTICS_RUNS_DIR", "/kaggle/working/runs")
os.environ.setdefault("ULTRALYTICS_OFFLINE", "True")
os.environ.setdefault("ULTRALYTICS_TELEMETRY", "False")
# Tắt mọi plotting/caching Ultralytics
os.environ.setdefault("YOLO_VERBOSE", "False")
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")


In [ ]:
# Kaggle /kaggle/working là output dir. Tất cả artifact phải vào đây.
import os
from pathlib import Path
WORKDIR = Path(os.environ.get("WORKDIR", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
print("workdir:", WORKDIR)


In [ ]:
def preflight(audit_dir: str) -> dict:
    """Kiểm tra manifest + hash + group-disjoint trước khi train.

    Trả về dict với 'ok' bool + 'pending_data' nếu thiếu.
    Raise nếu hash khác (file đã bị sửa sau export).
    """
    import hashlib
    import os
    import json

    manifest_path = os.path.join(audit_dir, "manifest.json")
    if not os.path.isfile(manifest_path):
        return {"ok": False, "pending_data": True,
                "reason": f"manifest.json không có tại {audit_dir}"}
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    # Manifest.sha256 đi kèm file đã verify ở R8; ở đây chỉ in lại
    digest_path = os.path.join(audit_dir, "manifest.sha256")
    if os.path.isfile(digest_path):
        recorded = open(digest_path, encoding="ascii").read().strip()
        actual = hashlib.sha256(open(manifest_path, "rb").read()).hexdigest()
        if recorded != actual:
            raise ValueError(
                f"manifest hash mismatch: file={recorded} vs recompute={actual}"
            )
    counts = manifest.get("counts", {})
    needed = {"train", "val", "test"}
    if not needed.issubset(counts):
        return {"ok": False, "pending_data": True,
                "reason": f"split thiếu: cần {needed}, có {set(counts)}"}
    if any(v == 0 for v in counts.values()):
        return {"ok": False, "pending_data": True,
                "reason": f"split có count=0: {counts}"}
    if manifest.get("holdout_status") != "human_curated_group_disjoint":
        return {"ok": False, "pending_data": True,
                "reason": f"holdout_status chưa phải human_curated: {manifest.get('holdout_status')}"}
    return {"ok": True, "manifest": manifest}


print("preflight helpers loaded")


In [ ]:
import os
from pathlib import Path
KAGGLE_INPUT = "/kaggle/input"
DATASET_SLUG = os.environ.get("PLATE_DATASET_SLUG", "plate-detect-v1")
audit_dir = os.path.join(KAGGLE_INPUT, DATASET_SLUG)
pf = preflight(audit_dir)
if not pf["ok"]:
    raise SystemExit(f"pending_data: {pf.get('reason', 'unknown')}")
manifest = pf["manifest"]
print(f"OK manifest: counts={manifest['counts']} holdout={manifest['holdout_status']}")
data_yaml = os.path.join(audit_dir, "data.yaml")
assert os.path.isfile(data_yaml), f"data.yaml không có tại {data_yaml}"


In [ ]:
from ultralytics import YOLO
# Pin YOLO11n — model size chọn theo handoff R10 (n trước, s sau khi có số đo).
model = YOLO("yolo11n.pt")  # Kaggle tự download hoặc user attach model
save_dir = WORKDIR / "runs" / "plate_detector_v1"
save_dir.mkdir(parents=True, exist_ok=True)

# Config từ manifest counts; epochs/imgsz theo handoff R6/R10 — điều chỉnh sau khi có lệch holdout
EPOCHS = int(os.environ.get("EPOCHS", "60"))
IMGSZ = int(os.environ.get("IMGSZ", "640"))
BATCH = int(os.environ.get("BATCH", "16"))

results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    save=True,
    save_period=-1,  # chỉ giữ best/last
    cache=False,    # không cache dataset vào RAM (giữ output gọn)
    plots=False,    # không plot (tiết kiệm disk)
    project=str(WORKDIR / "runs"),
    name="plate_detector_v1",
    exist_ok=True,
    resume=False,   # nếu muốn resume, dùng cell bên dưới
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=False,
    seed=42,
)
print("train done:", results.model_type if hasattr(results, "model_type") else "ok")


In [ ]:
# Resume từ last.pt (Ultralytics 8.4 hỗ trợ qua `resume=True` với `model.last`)
import os
from ultralytics import YOLO
last_ckpt = WORKDIR / "runs" / "plate_detector_v1" / "weights" / "last.pt"
if last_ckpt.is_file():
    model = YOLO(str(last_ckpt))
    results = model.train(
        data=data_yaml,
        resume=True,
        cache=False,
        plots=False,
        device=0 if torch.cuda.is_available() else "cpu",
    )
    print("resumed from", last_ckpt)
else:
    print(f"no checkpoint at {last_ckpt}, skipping resume")


In [ ]:
import hashlib
import shutil
from pathlib import Path
# Export best.pt → ONNX để import về runtime (R7)
best = WORKDIR / "runs" / "plate_detector_v1" / "weights" / "best.pt"
if not best.is_file():
    raise FileNotFoundError(f"best.pt không có tại {best}")
exported = YOLO(str(best)).export(format="onnx", imgsz=IMGSZ, opset=12,
                                  simplify=True, dynamic=False)
# exported là str path tới file .onnx
shutil.copy(exported, str(WORKDIR / "plate_detect_v1.onnx"))

# Manifest + hash
artifact = {
    "schema_version": 1,
    "task": "plate_detector",
    "engine": "ultralytics.yolo11",
    "package_versions": {
        "ultralytics": __import__("importlib.metadata").metadata.version("ultralytics"),
        "torch": torch.__version__,
    },
    "weights": {"best": str(best), "last": str(WORKDIR / "runs" / "plate_detector_v1" / "weights" / "last.pt")},
    "onnx": str(WORKDIR / "plate_detect_v1.onnx"),
    "class_mapping": manifest.get("names", {}),
    "holdout_status": manifest["holdout_status"],
    "config_sha256": hashlib.sha256(open(data_yaml, "rb").read()).hexdigest(),
    "source_manifest_sha256": manifest.get("source_manifest_sha256", ""),
}
artifact_path = WORKDIR / "artifact_manifest.json"
artifact_path.write_text(__import__("json").dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
sha = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
(WORKDIR / "artifact_manifest.sha256").write_text(sha, encoding="ascii")
print("artifact:", artifact_path, sha)


In [ ]:
# Round-trip smoke: predict trên 1 ảnh mẫu từ test split
from pathlib import Path
test_imgs = sorted((Path(audit_dir) / "images" / "test").glob("*.jpg"))
if not test_imgs:
    print("no test images; skip smoke")
else:
    sample = test_imgs[0]
    res = YOLO(str(WORKDIR / "plate_detect_v1.onnx")).predict(
        source=str(sample), imgsz=IMGSZ, conf=0.25, verbose=False,
    )
    n_boxes = len(res[0].boxes) if res else 0
    print(f"smoke: {sample.name} → {n_boxes} boxes")
    assert n_boxes > 0, f"smoke fail: {sample.name} không có box nào"
print("smoke ok")
